# Module 7.3: Custom Business Metrics for RAG Systems

**Level 2 - Distributed Tracing & Advanced Observability**

## Overview

This module teaches how to track RAG-specific business metrics beyond technical monitoring, bridging the gap between infrastructure observability and executive decision-making.

### Key Concepts

**Technical vs Business Metrics:**
- Technical metrics: latency, errors, uptime ("is the system working?")
- Business metrics: satisfaction, feature adoption, revenue attribution ("is it creating value?")

**Why This Matters:**
Executives need satisfaction scores and feature adoption data, not just latency percentiles. Technical metrics tell you HOW FAST things run, not HOW VALUABLE they are.

### Prerequisites

- Level 1 M2.3: Prometheus/Grafana
- Module 7.1: Distributed Tracing
- Module 7.2: APM Integration

### What We'll Build

1. RAG quality metrics (accuracy, satisfaction, hallucination rate)
2. User cohort analysis (FREE, PAID, ENTERPRISE, etc.)
3. Feature usage tracking
4. Executive KPI dashboard
5. Grafana dashboard implementation

## Purpose
Track business‑level KPIs for RAG (feature adoption, satisfaction, revenue attribution) and connect them to technical telemetry.

## Concepts Covered
Cohorts, feature usage, satisfaction & hallucination metrics, KPI aggregation, Prometheus export, offline‑first testability.

## After Completing
You can instrument/compute business KPIs for RAG, expose them via API/Prometheus, and validate them with smoke tests.

## Context in Track
L2 → Observability & Tracing; this module complements technical metrics with business value signals before L3 scaling.

---

## Section 1: Setup and Imports

Let's start by importing our module and loading configuration.

In [ ]:
# Import core modules
import json
import sys
from datetime import datetime
from pathlib import Path

# Add parent directory to path for imports
sys.path.insert(0, str(Path('..').resolve()))

from src.m7_custom_business_metrics import core as metrics
from src.m7_custom_business_metrics.config import load_config

# Load configuration
config = load_config()
print(f"✓ Configuration loaded")
print(f"  Environment: {config.environment}")
print(f"  Prometheus enabled: {config.prometheus.enabled}")
print(f"  Redis enabled: {config.redis.enabled}")
print(f"  ClickHouse enabled: {config.clickhouse.enabled}")

# Expected: Configuration status printed

## Section 2: RAG Quality Metrics

Track custom Prometheus metrics for RAG query outcomes:
- **Query accuracy**: accurate, partial, inaccurate, hallucinated
- **User satisfaction**: 1-5 scale ratings
- **Hallucination rate**: Percentage of hallucinated responses
- **Model confidence**: 0.0-1.0 scores

**Key Principle**: Use bounded labels (fixed cohort values), NOT unbounded user IDs. Using user IDs as labels causes Prometheus storage explosion!

In [ ]:
# Example: Record quality metrics for a single query
query_metrics = metrics.QueryMetrics(
    query_id="demo_q001",
    user_id="demo_u001",
    cohort=metrics.UserCohort.PAID,
    accuracy=metrics.QueryAccuracy.ACCURATE,
    satisfaction=5,
    confidence=0.92,
    feature=metrics.FeatureType.SIMPLE_QA,
    timestamp=datetime.utcnow(),
    latency_ms=234.5
)

metrics.record_query_metrics(query_metrics)
print(f"✓ Recorded metrics for query {query_metrics.query_id}")
print(f"  Cohort: {query_metrics.cohort.value}")
print(f"  Accuracy: {query_metrics.accuracy.value}")
print(f"  Satisfaction: {query_metrics.satisfaction}/5")

# Expected: Metrics recorded successfully

## Section 3: User Cohort Analysis

Define user segments for business analysis:
- **FREE, PAID, ENTERPRISE**: Subscription tiers
- **NEW**: First 30 days
- **POWER**: >100 queries/month
- **AT_RISK**: No queries in 14 days

**Critical Performance Requirement**: `get_user_cohort()` must execute in <10ms per query. Use Redis caching or nightly precomputation to avoid database lookups on the critical path.

In [ ]:
# Demonstrate cohort determination for different user types
test_users = [
    ("enterprise_user", {"tier": "enterprise", "days_since_signup": 100, "query_count": 500, "days_since_last_query": 1}),
    ("new_user", {"tier": "free", "days_since_signup": 10, "query_count": 5, "days_since_last_query": 0}),
    ("power_user", {"tier": "free", "days_since_signup": 60, "query_count": 150, "days_since_last_query": 1}),
    ("at_risk_user", {"tier": "paid", "days_since_signup": 200, "query_count": 80, "days_since_last_query": 20}),
]

print("Cohort Determination Results:")
for user_id, metadata in test_users:
    cohort = metrics.get_user_cohort(user_id, metadata)
    print(f"  {user_id:20s} → {cohort.value}")

# Expected: Different cohorts assigned based on user characteristics

## Section 4: Hallucination Rate & Cohort Metrics

Track critical business metrics per cohort:
- **Hallucination rate**: Percentage of queries with hallucinated responses
- **Active users**: Current active user count per cohort
- **Cost tracking**: Cumulative cost per cohort

These metrics help identify which user segments have quality issues and understand cost distribution.

In [ ]:
# Update hallucination rate for FREE cohort
rate = metrics.update_hallucination_rate(
    cohort=metrics.UserCohort.FREE,
    total_queries=1000,
    hallucinated_queries=15
)
print(f"✓ Hallucination rate for FREE: {rate:.2f}%")

# Update active users per cohort
metrics.update_active_users(metrics.UserCohort.FREE, 150)
metrics.update_active_users(metrics.UserCohort.PAID, 75)
metrics.update_active_users(metrics.UserCohort.ENTERPRISE, 25)
print(f"✓ Active users updated")

# Record costs per cohort (based on query volume)
metrics.record_cost(metrics.UserCohort.FREE, 5.00)
metrics.record_cost(metrics.UserCohort.PAID, 22.50)
metrics.record_cost(metrics.UserCohort.ENTERPRISE, 75.00)
print(f"✓ Costs recorded")

# Expected: Metrics updated for each cohort

## Section 5: Feature Usage Tracking

Monitor adoption of RAG capabilities:
- **simple_qa**: Basic question-answering
- **summarization**: Document summarization
- **multi_doc**: Multi-document analysis
- **conversational**: Conversational interactions

**Why This Matters**: Identify which features actually drive user engagement vs. features that are built but unused. This informs product prioritization.

In [ ]:
# Update feature success rates
features_data = [
    (metrics.FeatureType.SIMPLE_QA, 700, 595),  # 85% success
    (metrics.FeatureType.SUMMARIZATION, 200, 180),  # 90% success
    (metrics.FeatureType.MULTI_DOC, 100, 82),  # 82% success
    (metrics.FeatureType.CONVERSATIONAL, 150, 120),  # 80% success
]

print("Feature Success Rates:")
for feature, total, successful in features_data:
    rate = metrics.update_feature_success_rate(feature, total, successful)
    print(f"  {feature.value:20s}: {rate:.1f}% ({successful}/{total})")

# Expected: Success rates calculated and displayed

## Section 6: Executive KPI Calculations

Aggregate raw metrics into business KPIs that executives understand:
- **Cost per user**: Infrastructure cost divided by active users
- **Satisfaction trend**: Direction and magnitude of satisfaction changes
- **Feature adoption rate**: Percentage of queries using each feature
- **Cohort retention**: Percentage of users retained over time

These KPIs translate technical measurements into business impact.

In [ ]:
# Calculate cost per user
total_cost = 102.50  # From cohort costs above
total_users = 250
cpu = metrics.calculate_cost_per_user(total_cost, total_users)
print(f"Cost per user: ${cpu:.4f}")

# Calculate satisfaction trend (mock data)
satisfaction_scores = [3.5, 3.6, 3.8, 3.9, 4.0, 4.1, 4.2]
trend_pct, direction = metrics.calculate_satisfaction_trend(satisfaction_scores)
print(f"Satisfaction trend: {direction} ({trend_pct:+.1f}%)")

# Calculate feature adoption rates
adoption = metrics.calculate_feature_adoption_rate(
    {"simple_qa": 700, "summarization": 200, "multi_doc": 100, "conversational": 150},
    total_queries=1150
)
print(f"Feature adoption rates:")
for feature, rate in adoption.items():
    print(f"  {feature:20s}: {rate:.1f}%")

# Expected: Executive KPIs calculated

## Section 7: Common Failures & How to Fix Them

### Failure 1: Cardinality Explosion
**What happens**: Using unbounded labels (user_id, query_id) causes Prometheus storage to explode.

**Fix**: Use bounded cohort labels instead of individual IDs.

In [ ]:
# Demonstrate cardinality explosion detection
safe_labels = [f"cohort_{i}" for i in range(6)]  # 6 cohorts - SAFE
unsafe_labels = [f"user_{i}" for i in range(10000)]  # 10K users - UNSAFE

is_safe = metrics.handle_cardinality_explosion(safe_labels, max_cardinality=100)
print(f"Bounded cohorts (n={len(safe_labels)}): {'✓ Safe' if is_safe else '✗ Unsafe'}")

is_safe = metrics.handle_cardinality_explosion(unsafe_labels, max_cardinality=100)
print(f"Unbounded user IDs (n={len(unsafe_labels)}): {'✓ Safe' if is_safe else '✗ Unsafe'}")

# Validate label structure
valid = metrics.validate_metric_labels({"cohort": "paid", "feature": "simple_qa"})
print(f"Valid labels: {'✓' if valid else '✗'}")

invalid = metrics.validate_metric_labels({"user_id": "u123", "feature": "simple_qa"})
print(f"Invalid labels (unbounded): {'✓' if invalid else '✗'}")

# Expected: Cardinality explosion detected for unsafe labels

# Load example data (offline-first: fallback to sample data if file not found)
try:
    with open('../data/example_data.json', 'r') as f:
        data = json.load(f)
    print(f"Loaded {len(data['queries'])} example queries")
except FileNotFoundError:
    print("⚠️ Example data file not found, using inline sample data")
    data = {
        'queries': [
            {
                'query_id': 'q001', 'user_id': 'u101',
                'user_metadata': {'tier': 'free', 'days_since_signup': 45, 'query_count': 25, 'days_since_last_query': 2},
                'accuracy': 'accurate', 'satisfaction': 5, 'confidence': 0.92,
                'feature': 'simple_qa', 'latency_ms': 234.5
            },
            {
                'query_id': 'q002', 'user_id': 'u102',
                'user_metadata': {'tier': 'paid', 'days_since_signup': 120, 'query_count': 450, 'days_since_last_query': 1},
                'accuracy': 'accurate', 'satisfaction': 4, 'confidence': 0.88,
                'feature': 'summarization', 'latency_ms': 456.2
            },
            {
                'query_id': 'q003', 'user_id': 'u103',
                'user_metadata': {'tier': 'enterprise', 'days_since_signup': 180, 'query_count': 1250, 'days_since_last_query': 0},
                'accuracy': 'accurate', 'satisfaction': 5, 'confidence': 0.95,
                'feature': 'multi_doc', 'latency_ms': 892.3
            }
        ]
    }

# Process first 3 queries to demonstrate
for query_data in data['queries'][:3]:
    cohort = metrics.get_user_cohort(
        query_data['user_id'],
        query_data['user_metadata']
    )
    
    query = metrics.QueryMetrics(
        query_id=query_data['query_id'],
        user_id=query_data['user_id'],
        cohort=cohort,
        accuracy=metrics.QueryAccuracy(query_data['accuracy']),
        satisfaction=query_data.get('satisfaction'),
        confidence=query_data['confidence'],
        feature=metrics.FeatureType(query_data['feature']),
        timestamp=datetime.utcnow(),
        latency_ms=query_data['latency_ms']
    )
    
    metrics.record_query_metrics(query)

print(f"✓ Processed example queries")

# Expected: Example data loaded and processed

In [ ]:
# Load example data
with open('../data/example_data.json', 'r') as f:
    data = json.load(f)

print(f"Loaded {len(data['queries'])} example queries")

# Process first 3 queries to demonstrate
for query_data in data['queries'][:3]:
    cohort = metrics.get_user_cohort(
        query_data['user_id'],
        query_data['user_metadata']
    )
    
    query = metrics.QueryMetrics(
        query_id=query_data['query_id'],
        user_id=query_data['user_id'],
        cohort=cohort,
        accuracy=metrics.QueryAccuracy(query_data['accuracy']),
        satisfaction=query_data.get('satisfaction'),
        confidence=query_data['confidence'],
        feature=metrics.FeatureType(query_data['feature']),
        timestamp=datetime.utcnow(),
        latency_ms=query_data['latency_ms']
    )
    
    metrics.record_query_metrics(query)

print(f"✓ Processed example queries")

# Expected: Example data loaded and processed

## Section 9: Decision Framework

### When to Use Prometheus + Grafana

**✓ Best for:**
- RAG systems under 100K queries/month
- Quick business visibility without BI platform overhead
- Teams already using Prometheus for technical metrics
- Limited analytics budget (<$100/month)

### Alternative Solutions

**Product Analytics Platforms** (Mixpanel, Amplitude)
- Better for: User journey analysis, detailed funneling, cohort experiments
- Cost: $200-1000+/month
- Trade-off: More features but higher cost

**BI Tools** (Looker, Tableau)
- Better for: Complex ad-hoc queries, executive reporting
- Cost: $300-2000+/month + data warehouse
- Trade-off: Superior analytics but requires infrastructure

**Manual Reports**
- Better for: <100 users, pre-product-market fit
- Cost: Time only
- Trade-off: Free but doesn't scale

### When NOT to Use This Approach

- Under 100 users without product-market fit (use manual reports)
- Need detailed user funnel/journey analysis (use product analytics)
- Scaling beyond 100K queries/month without infrastructure investment

In [ ]:
# Generate executive summary
summary = metrics.generate_executive_summary(time_period="last_7_days")

print(f"Executive Summary for {summary['period']}:")
print(f"  Generated: {summary['generated_at']}")
print(f"  Total queries: {summary['kpis']['total_queries']}")
print(f"  Active users: {summary['kpis']['active_users']}")
print(f"  Avg satisfaction: {summary['kpis']['avg_satisfaction']:.2f}/5.0")
print(f"  Satisfaction trend: {summary['kpis']['satisfaction_trend']}")

# Expected: Executive summary generated with KPI structure

## Summary & Key Takeaways

### What We Built

1. ✓ **RAG Quality Metrics**: Tracked accuracy, satisfaction, hallucination rate, and confidence
2. ✓ **User Cohort Analysis**: Segmented users into FREE, PAID, ENTERPRISE, NEW, POWER, AT_RISK
3. ✓ **Feature Usage Tracking**: Monitored adoption of RAG capabilities
4. ✓ **Executive KPI Dashboard**: Aggregated metrics into business-relevant KPIs
5. ✓ **Failure Handling**: Detected cardinality explosions and label validation issues

### Critical Insights

**Technical metrics tell you HOW FAST things run, not HOW VALUABLE they are.**

- Use bounded labels (cohorts) not unbounded IDs
- Cohort lookups must be <10ms (use Redis cache)
- Prometheus + Grafana works for <100K queries/month
- Product analytics platforms excel at user journey analysis
- Always validate metric labels to prevent storage explosion

### Production Checklist

Before deploying to production:

1. ☐ Configure Redis for fast cohort lookups
2. ☐ Set Prometheus retention policies
3. ☐ Create Grafana dashboards for executives
4. ☐ Set up alerts for hallucination rate thresholds
5. ☐ Test cardinality limits on metric labels
6. ☐ Document cohort definitions for stakeholders

### Next Steps

- Module 7.4: Log Aggregation with ELK Stack
- Module 7.5: Real-time Alerting
- Practathon Challenge: Build complete KPI system with alerting (4-5 hours)